# 🎨 Snap GenAI Lens - Colab Demo

**Identity-Preserving Face Stylization for AR Lenses**

This notebook demonstrates a production-ready GenAI lens system that:
- Detects faces and extracts landmarks
- Generates multiple conditioning signals
- Uses Stable Diffusion + ControlNet for stylization
- Preserves identity during transformation
- Measures quality and performance metrics

---

## 🚀 Quick Start

1. **Runtime**: Change runtime to GPU (Runtime → Change runtime type → GPU)
2. **Run all cells** (Runtime → Run all)
3. **Upload your selfie** when the interface appears
4. **Generate!**

## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install torch torchvision diffusers transformers accelerate
!pip install mediapipe insightface onnxruntime
!pip install opencv-python pillow scikit-image
!pip install controlnet-aux
!pip install gradio
!pip install matplotlib tqdm

print("✓ All dependencies installed!")

## 📥 Step 2: Clone Repository

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/snap-genai-lens.git
%cd snap-genai-lens

print("✓ Repository cloned!")

## 🎯 Step 3: Quick Test - Face Detection

In [ ]:
import cv2
import numpy as np
from preprocessing.face_processor import FacePreprocessor
import matplotlib.pyplot as plt

# Initialize face processor
face_processor = FacePreprocessor()

# Test with sample image
# You can upload your own image here
from google.colab import files
uploaded = files.upload()

# Process first uploaded image
filename = list(uploaded.keys())[0]
image = cv2.imread(filename)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Process
results = face_processor.process_image(image_rgb)

if results['success']:
    print("✅ Face detected!")
    print(f"Confidence: {results['face_info']['confidence']:.3f}")
    print(f"Landmarks: {len(results['landmarks'])} points")
    
    # Visualize
    viz = face_processor.visualize_preprocessing(image_rgb, results)
    
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb)
    plt.title('Original')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(viz)
    plt.title('Face Detection')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"❌ {results['error']}")

## 🎨 Step 4: Test Conditioning Generation

In [ ]:
from conditioning.condition_generator import ConditioningGenerator

# Initialize
cond_gen = ConditioningGenerator()

# Generate all conditioning signals
conditions = cond_gen.generate_all_conditions(
    image_rgb,
    results,
    use_canny=True,
    use_landmarks=True,
    use_depth=False,
    use_mask=True
)

# Visualize
viz_grid = cond_gen.visualize_conditions(conditions)

plt.figure(figsize=(15, 5))
plt.imshow(viz_grid)
plt.title('Conditioning Signals')
plt.axis('off')
plt.show()

print(f"✓ Generated {len(conditions)} conditioning signals")

## 🤖 Step 5: Load Models & Generate

In [ ]:
from models.inference import StyleLensInference

# Initialize inference pipeline
print("Loading Stable Diffusion + ControlNet...")
inference = StyleLensInference(
    model_id="runwayml/stable-diffusion-v1-5",
    controlnet_id="lllyasviel/sd-controlnet-canny",
    use_fp16=True
)

# Load model
inference.load_model()

print("\n✓ Models loaded!")
print(f"Device: {inference.device}")
print(f"Dtype: {inference.dtype}")

## ✨ Step 6: Generate Stylized Image

In [ ]:
# Combine conditioning signals
weights = {
    'canny': 0.6,
    'landmarks': 0.4,
    'mask': 0.3
}

combined_cond = cond_gen.combine_conditions(conditions, weights)

# Generate!
print("Generating styled image...")
result = inference.generate(
    conditioning_image=combined_cond,
    style='anime',  # Try: anime, cyberpunk, sketch, oil_painting
    num_inference_steps=20,
    guidance_scale=7.5,
    seed=42
)

if result['success']:
    print(f"\n✅ Generation successful!")
    print(f"Inference time: {result['inference_time']:.2f}s")
    print(f"FPS: {1/result['inference_time']:.2f}")
    
    # Display
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(image_rgb)
    plt.title('Original')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(combined_cond)
    plt.title('Conditioning')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(result['image'])
    plt.title(f"Generated ({result['metadata']['style']})") 
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"❌ Generation failed: {result.get('error')}")

## 📊 Step 7: Evaluate Results

In [ ]:
from identity.identity_preserver import IdentityPreserver
from evaluation.evaluator import LensEvaluator

# Initialize evaluators
identity_preserver = IdentityPreserver()
identity_preserver.load_model()

evaluator = LensEvaluator()

# Measure identity preservation
identity_metrics = identity_preserver.measure_identity_preservation(
    image_rgb,
    result['image'],
    results['face_info']['bbox']
)

print("\n📊 EVALUATION RESULTS")
print("=" * 50)
print(f"\nIdentity Similarity: {identity_metrics['identity_similarity']:.3f}")
print(f"Identity Preserved: {identity_metrics['identity_preserved']}")
print(f"Confidence: {identity_metrics['confidence']}")

# Complete evaluation
eval_results = evaluator.evaluate_generation(
    original_image=image_rgb,
    generated_image=result['image'],
    conditioning_image=combined_cond,
    style_prompt=result['metadata']['prompt'],
    identity_metrics=identity_metrics,
    inference_time=result['inference_time']
)

if 'clip_similarity' in eval_results:
    print(f"\nCLIP Similarity: {eval_results['clip_similarity']:.3f}")

if 'overall_quality' in eval_results:
    print(f"Overall Quality Score: {eval_results['overall_quality']:.3f}")

print(f"\nInference Time: {result['inference_time']:.2f}s")
print(f"FPS: {eval_results.get('fps', 0):.2f}")

## 🎯 Step 8: Launch Interactive Demo

In [ ]:
# Launch Gradio interface
!python app.py

## 🧪 Step 9: Benchmark Performance

In [ ]:
# Run benchmark at different step counts
benchmark_results = inference.benchmark(
    conditioning_image=combined_cond,
    steps_list=[10, 15, 20, 30, 50]
)

# Display results
import pandas as pd

df = pd.DataFrame(benchmark_results)
print("\n📈 PERFORMANCE BENCHMARK")
print("=" * 50)
print(df.to_string(index=False))

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df['steps'], df['time'], marker='o', linewidth=2, markersize=8)
plt.xlabel('Diffusion Steps')
plt.ylabel('Inference Time (s)')
plt.title('Performance vs Quality Tradeoff')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Recommendations
print("\n💡 RECOMMENDATIONS")
print("=" * 50)
print("For real-time use: 10-15 steps (~0.5-1s)")
print("For quality: 20-30 steps (~1-2s)")
print("For maximum quality: 50 steps (~3-4s)")

## 🎨 Step 10: Try Different Styles

In [ ]:
# Generate with multiple styles
styles = ['anime', 'cyberpunk', 'sketch', 'oil_painting']
results_dict = {}

print("Generating multiple styles...\n")

for style in styles:
    print(f"Generating {style} style...")
    gen_result = inference.generate(
        conditioning_image=combined_cond,
        style=style,
        num_inference_steps=20,
        seed=42
    )
    
    if gen_result['success']:
        results_dict[style] = gen_result['image']
        print(f"  ✓ {gen_result['inference_time']:.2f}s")

# Display grid
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Original
axes[0].imshow(image_rgb)
axes[0].set_title('Original', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Conditioning
axes[1].imshow(combined_cond)
axes[1].set_title('Conditioning', fontsize=14, fontweight='bold')
axes[1].axis('off')

# Generated styles
for idx, (style, img) in enumerate(results_dict.items(), start=2):
    axes[idx].imshow(img)
    axes[idx].set_title(style.replace('_', ' ').title(), fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Generated {len(results_dict)} styles!")

## 💾 Step 11: Save Results

In [ ]:
import os
from PIL import Image

# Create output directory
os.makedirs('outputs', exist_ok=True)

# Save generated images
for style, img in results_dict.items():
    pil_img = Image.fromarray(img)
    pil_img.save(f'outputs/generated_{style}.png')

# Save comparison
comparison = evaluator.create_comparison_grid(
    original=image_rgb,
    conditioning=combined_cond,
    generated=results_dict['anime'],
    metrics=eval_results
)

Image.fromarray(comparison).save('outputs/comparison.png')

print("✓ Results saved to 'outputs/' directory")

# Download all outputs
!zip -r outputs.zip outputs/
files.download('outputs.zip')

## 🎓 Conclusion

### ✅ What This Project Demonstrates

1. **Face-Aware AI**: Multi-modal face detection and landmark extraction
2. **Compositional Generation**: Multiple conditioning signals for better control
3. **Identity Preservation**: Face embedding similarity measurement
4. **Production Thinking**: Latency optimization, failure handling, metrics
5. **System Design**: Modular architecture, clean interfaces

### 🎯 Snap MLE Interview Talking Points

- **AR + GenAI**: Face-aware conditioning maps directly to Snap Lenses
- **Mobile Constraints**: Optimized for 15-20 steps, FP16, ~1s inference
- **Quality Metrics**: CLIP, identity similarity, human evaluation
- **Failure Handling**: Explicit face detection confidence checks
- **Scalability**: Discusses how this would work in production

### 📚 Next Steps

1. Add temporal consistency for video
2. Integrate IP-Adapter for stronger identity preservation  
3. A/B test with human raters
4. Optimize with distillation/quantization
5. Deploy as API endpoint

---

**Made for Snap Graduate MLE – GenAI Role** 🚀